In [3]:
# 07_DOCUMENTATION — PART1 / CELL 01
# REPOSITORY INVENTORY & DOCUMENTATION BLUEPRINT
#
# Exact Git HEAD | tracked-file inventory | SHA256
# Read-only research sources | No overwrite
# No retraining | Self-contained (stdlib + pandas)
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
from collections import Counter
import subprocess
import hashlib
import json
import os
import pandas as pd

ROOT = Path(r"C:\Penelitian Sentimen Bencana Nasional")
OUT = ROOT / "07_Documentation" / "outputs" / "repository_inventory"

EXPECTED_HEAD = (
    "ae5107c317eaee11a909251470be4b7503a6f9db"
)

AUDIT6 = (
    ROOT / "06_Result" / "outputs"
    / "final_technical_report"
    / "part6_final_technical_audit.json"
)

INVENTORY_OUT = OUT / "repository_inventory.csv"
SUMMARY_OUT = OUT / "repository_inventory_summary.json"
BLUEPRINT_OUT = OUT / "documentation_blueprint.md"
AUDIT_OUT = OUT / "part1_repository_inventory_audit.json"

OUTPUTS = [
    INVENTORY_OUT,
    SUMMARY_OUT,
    BLUEPRINT_OUT,
    AUDIT_OUT
]

EXPECTED_PHASES = [
    "04_Modeling",
    "05_Temporal",
    "06_Result"
]

def sha256(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(
            lambda: f.read(1024 * 1024), b""
        ):
            h.update(block)
    return h.hexdigest()

def git_bytes(*args):
    p = subprocess.run(
        ["git", "-C", str(ROOT), *args],
        check=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE
    )
    return p.stdout

def git_text(*args):
    return git_bytes(*args).decode("utf-8").strip()

def json_bytes(data):
    return json.dumps(
        data, ensure_ascii=False, indent=2,
        allow_nan=False
    ).encode("utf-8")

def phase_of(path):
    p = Path(path)
    first = p.parts[0] if p.parts else ""
    if first in {
        "01_Raw Data", "01_Data_Collection",
        "01_DataCollection"
    }:
        return "01_Collection"
    if first.startswith("01_"):
        return "01_Collection"
    if first.startswith("02_"):
        return "02_Preprocessing"
    if first.startswith("03_"):
        return "03_Labeling"
    if first.startswith("04_"):
        return "04_Modeling"
    if first.startswith("05_"):
        return "05_Temporal"
    if first.startswith("06_"):
        return "06_Result"
    if first.startswith("07_"):
        return "07_Documentation"
    return "ROOT_OR_OTHER"

def artifact_type(path):
    suffix = Path(path).suffix.lower()
    return {
        ".ipynb": "notebook",
        ".csv": "dataset_or_table",
        ".json": "json_manifest_or_audit",
        ".md": "markdown_document",
        ".pdf": "pdf_report",
        ".png": "figure_png",
        ".jpg": "figure_jpg",
        ".jpeg": "figure_jpg",
        ".py": "python_script",
        ".txt": "text_or_config",
        ".toml": "environment_config",
        ".yml": "environment_config",
        ".yaml": "environment_config",
        ".lock": "dependency_lock"
    }.get(suffix, "other")

def role_of(path):
    lower = path.lower()
    if lower.endswith(".ipynb"):
        return "execution_notebook"
    if "audit" in lower:
        return "audit_evidence"
    if "manifest" in lower or "registry" in lower:
        return "provenance_or_manifest"
    if lower.endswith((".png", ".jpg", ".jpeg")):
        return "research_figure"
    if lower.endswith(".pdf"):
        return "report_or_reference"
    if lower.endswith(".csv"):
        return "dataset_or_results_table"
    if lower.endswith(".json"):
        return "structured_research_artifact"
    if lower.endswith(".md"):
        return "research_documentation"
    return "other"

def read_json(path):
    return json.loads(path.read_text(encoding="utf-8"))

print("=" * 76)
print("07_DOCUMENTATION PART1 — REPOSITORY PREFLIGHT")
print("=" * 76)

assert ROOT.is_dir(), f"Root missing: {ROOT}"
assert (ROOT / ".git").exists(), "Git repository not found."
assert AUDIT6.is_file(), "Result Part6 audit missing."

for path in OUTPUTS:
    if path.exists():
        raise FileExistsError(
            f"Refusing overwrite: {path}"
        )

# ------------------------------------------------------------
# A. GIT COMMIT LOCK
# ------------------------------------------------------------

head = git_text("rev-parse", "HEAD")
branch = git_text("branch", "--show-current")

assert head == EXPECTED_HEAD, (
    "Git HEAD differs from audited commit.\n"
    f"Expected: {EXPECTED_HEAD}\nActual: {head}\n"
    "Review repository changes before continuing."
)

# Normalize Windows paths before comparing Git repository roots.
git_root = Path(
    git_text("rev-parse", "--show-toplevel")
).resolve()

expected_root = ROOT.resolve()

assert os.path.normcase(str(git_root)) == os.path.normcase(
    str(expected_root)
), (
    "Git repository root mismatch.\n"
    f"Git root : {git_root}\n"
    f"Expected : {expected_root}"
)

print("GIT REPOSITORY ROOT: PASS")
print("Repository:", git_root)

# Worktree changes to committed research source files are
# forbidden. The new documentation notebook may be untracked.
status_raw = git_bytes(
    "status", "--porcelain=v1", "-z", "--untracked-files=no"
)

changed_tracked = []

for entry in status_raw.split(b"\0"):
    if not entry:
        continue

    decoded = entry.decode("utf-8", errors="replace")
    changed_tracked.append(decoded)

# Git rename/copy porcelain can use two NUL entries.
# Rather than parsing ambiguously, fail closed for any
# tracked change.
assert not changed_tracked, (
    "Tracked files have uncommitted changes. "
    "Commit or restore them before inventory.\n"
    + "\n".join(changed_tracked[:20])
)

print("GIT HEAD LOCK: PASS")
print("Commit:", head)
print("Branch:", branch)

# ------------------------------------------------------------
# B. VALIDATE RESULT PART6 SOURCE
# ------------------------------------------------------------

a6 = read_json(AUDIT6)

assert a6["status"] == (
    "RESULT_PART6_TECHNICAL_AUDIT_PASS"
)

assert a6["dataset_rows"] == 348
assert a6["research_questions"] == 3
assert a6["original_figures_in_pdf"] == 10
assert a6["methodological_issues_open"] == 4
assert a6["academic_status"] == "PROVISIONAL"
assert a6["supervisor_approval_confirmed"] is False
assert a6["model_training_performed"] is False

for source_name, expected_hash in (
    a6["source_sha256"].items()
):
    path = ROOT / source_name

    assert path.is_file(), f"Missing locked source: {path}"
    assert sha256(path) == expected_hash, (
        f"Locked source modified: {source_name}"
    )

for image_name, expected_hash in (
    a6["original_figure_sha256"].items()
):
    path = ROOT / image_name

    assert path.is_file(), f"Missing original image: {path}"
    assert sha256(path) == expected_hash, (
        f"Original image modified: {image_name}"
    )

final_dir = AUDIT6.parent

for filename, expected_hash in (
    a6["output_sha256"].items()
):
    path = final_dir / filename
    assert path.is_file(), f"Missing Part6 output: {path}"
    assert sha256(path) == expected_hash

audit6_hash = sha256(AUDIT6)

print("RESULT PART6 & LOCKED RESEARCH SOURCES: PASS")

# ------------------------------------------------------------
# C. INVENTORY ALL TRACKED FILES AT EXACT HEAD
# ------------------------------------------------------------

tracked_raw = git_bytes(
    "ls-tree", "-r", "-z", "--full-tree", "HEAD"
)

records = []
seen_paths = set()

for entry in tracked_raw.split(b"\0"):
    if not entry:
        continue

    metadata, path_bytes = entry.split(b"\t", 1)
    mode, kind, git_oid = metadata.decode("ascii").split()

    relative_path = path_bytes.decode("utf-8")
    assert relative_path not in seen_paths
    seen_paths.add(relative_path)

    # Fail closed on submodules, symlinks, or unusual Git types.
    assert kind == "blob", (
        f"Unsupported Git tree item: {relative_path}"
    )
    assert mode in {"100644", "100755"}, (
        f"Unexpected Git mode {mode}: {relative_path}"
    )

    file_path = (ROOT / relative_path).resolve()

    assert file_path.is_relative_to(ROOT.resolve()), (
        f"Unsafe path: {relative_path}"
    )
    assert file_path.is_file(), (
        f"Tracked file missing locally: {relative_path}"
    )
    assert not file_path.is_symlink()

    file_size = file_path.stat().st_size

    records.append({
        "repository_path": relative_path,
        "phase": phase_of(relative_path),
        "artifact_type": artifact_type(relative_path),
        "role": role_of(relative_path),
        "extension": file_path.suffix.lower(),
        "size_bytes": file_size,
        "sha256": sha256(file_path),
        "git_mode": mode,
        "git_blob_oid": git_oid
    })

inventory = pd.DataFrame(records)
inventory = inventory.sort_values(
    "repository_path"
).reset_index(drop=True)

assert len(inventory) > 0
assert inventory["repository_path"].is_unique
assert inventory["sha256"].str.len().eq(64).all()

phase_counts = Counter(inventory["phase"])

for phase in EXPECTED_PHASES:
    assert phase_counts[phase] > 0, (
        f"No tracked files in {phase}"
    )

# The inventory represents the Git HEAD tree exactly.
assert len(inventory) == len(seen_paths)

print("TRACKED GIT TREE INVENTORY: PASS")
print("Total tracked files:", len(inventory))

# ------------------------------------------------------------
# D. COMPLETENESS & CRITICAL ARTIFACT VALIDATION
# ------------------------------------------------------------

required_paths = [
    "04_Modeling/04_modeling.ipynb",
    "05_Temporal/05_temporal.ipynb",
    "06_Result/06_result.ipynb",
    "04_Modeling/outputs/comparative_evaluation/"
        "paired_oof_comparison_348.csv",
    "05_Temporal/outputs/daily_aggregation/"
        "daily_sentiment_counts_20_days.csv",
    "06_Result/outputs/integrated_report/"
        "provisional_integrated_research_report.md",
    "06_Result/outputs/final_technical_report/"
        "provisional_integrated_research_report.pdf",
    "06_Result/outputs/final_technical_report/"
        "part6_final_technical_audit.json"
]

path_set = set(inventory["repository_path"])

for path in required_paths:
    assert path in path_set, (
        f"Critical tracked artifact missing: {path}"
    )

source_inventory = inventory.set_index("repository_path")

assert source_inventory.loc[
    "06_Result/outputs/final_technical_report/"
    "part6_final_technical_audit.json", "sha256"
] == audit6_hash

# ------------------------------------------------------------
# E. DOCUMENTATION BLUEPRINT
# ------------------------------------------------------------

summary = {
    "stage": "07_Documentation",
    "part": "1 Repository Inventory",
    "generated_at_utc":
        datetime.now(timezone.utc).isoformat(),
    "repository": (
        "YusakSR/Penelitian-Sentimen-Bencana-Nasional"
    ),
    "head_commit": head,
    "head_branch": branch,
    "tracked_file_count": int(len(inventory)),
    "tracked_total_size_bytes": int(
        inventory["size_bytes"].sum()
    ),
    "phase_file_counts": dict(
        sorted(Counter(inventory["phase"]).items())
    ),
    "artifact_type_counts": dict(
        sorted(Counter(inventory["artifact_type"]).items())
    ),
    "extension_counts": dict(
        sorted(Counter(inventory["extension"]).items())
    ),
    "critical_artifacts_verified": len(required_paths),
    "result_part6_audit_sha256": audit6_hash,
    "academic_status": "PROVISIONAL",
    "methodological_issues_open": 4,
    "full_git_tree_inventory": True,
    "model_training_performed": False
}

phase_lines = "\n".join(
    f"| {phase} | {count} |"
    for phase, count in sorted(phase_counts.items())
)

blueprint = f"""# 07_DOCUMENTATION — BLUEPRINT

**Repository:** {summary["repository"]}

**Commit terkunci:** `{head}`

**Status akademik:** PROVISIONAL

**Jumlah berkas terlacak:** {len(inventory)}

## 1. Prinsip Dokumentasi

Fase ini menjelaskan pipeline dan bukti penelitian
tanpa mengubah raw data, label, model, prediksi OOF,
grafik, atau laporan sumber.

Setiap part memakai input terkunci, audit SHA-256,
dan tidak menimpa output yang sudah diterbitkan.

## 2. Inventaris Repository

| Fase / kategori | Berkas |
|---|---:|
{phase_lines}

Daftar lengkap tersedia pada
`repository_inventory.csv`.

Inventaris ini merepresentasikan berkas terlacak
Git pada commit yang disebutkan di atas.
Berkas lokal belum terlacak tidak termasuk.

## 3. Rencana Part2–Part7

### Part2 — Data Provenance & Collection Documentation

Dokumentasikan aliran data dari pengumpulan,
pembersihan, deduplikasi, hingga 348 tweet
berlabel yang menjadi dataset penelitian.

Artefak:
- Data lineage
- Data dictionary
- Sampling and labeling documentation
- Provenance audit

### Part3 — Modeling & Temporal Reproducibility

Dokumentasikan:
- Python environment dan dependencies
- Seed dan SGKF 3-fold
- Tokenisasi dan training LSTM
- Baseline TF-IDF + Logistic Regression
- Prediksi OOF
- Agregasi tanggal WIB
- Sumber tabel dan grafik terkunci

Artefak:
- Reproducibility guide
- Configuration registry
- Source manifest
- Audit

### Part4 — Academic Methodology Alignment

Verifikasi kecocokan proposal, RM1–RM3,
dan implementasi.

Empat isu harus tetap terbuka kecuali terdapat
keputusan eksplisit dan terdokumentasi
dari pembimbing.

### Part5 — Figures, Tables & Reporting Guide

Susun katalog tabel, gambar, caption,
sumber bukti, dan pedoman editorial BAB IV.

### Part6 — End-to-End Repository Integrity Audit

Periksa rantai hash dan konsistensi artefak
tanpa melakukan retraining atau scraping ulang.

### Part7 — Final Documentation Handoff

Susun panduan repository, ringkasan pipeline,
manifest, serta paket penyerahan penelitian.

## 4. Gerbang Kualitas

- Semua artefak sumber harus dapat diverifikasi.
- Tidak ada penulisan ulang file sumber.
- Tidak ada angka atau rumusan masalah yang dikarang.
- Dataset penelitian tetap 348 tweet.
- Keempat isu metodologis tetap dilaporkan.
- PDF BAB IV tetap berstatus PROVISIONAL.
- Pemeriksaan visual PDF harus diselesaikan
  sebelum penyerahan akademik.
- Perubahan metode memerlukan persetujuan pembimbing.

## 5. Output Part1

- `repository_inventory.csv`
- `repository_inventory_summary.json`
- `documentation_blueprint.md`
- `part1_repository_inventory_audit.json`

**Tahap berikutnya:** Documentation Part2 —
Data Provenance & Collection Documentation.
"""

# ------------------------------------------------------------
# F. OUTPUT GENERATION — NO OVERWRITE
# ------------------------------------------------------------

output_bytes = {
    INVENTORY_OUT: inventory.to_csv(
        index=False
    ).encode("utf-8-sig"),
    SUMMARY_OUT: json_bytes(summary),
    BLUEPRINT_OUT: blueprint.encode("utf-8")
}

audit1 = {
    "stage": "07_Documentation",
    "part": "1 Repository Inventory & Blueprint",
    "generated_at_utc":
        datetime.now(timezone.utc).isoformat(),
    "head_commit": head,
    "head_branch": branch,
    "head_tree_sha": git_text(
        "rev-parse", "HEAD^{tree}"
    ),
    "tracked_files": int(len(inventory)),
    "tracked_total_size_bytes": int(
        inventory["size_bytes"].sum()
    ),
    "critical_artifacts_verified": len(required_paths),
    "result_part6_status": a6["status"],
    "result_part6_audit_sha256": audit6_hash,
    "dataset_rows": 348,
    "methodological_issues_open": 4,
    "academic_status": "PROVISIONAL",
    "sources_modified": False,
    "model_training_performed": False,
    "output_sha256": {
        path.name: hashlib.sha256(data).hexdigest()
        for path, data in output_bytes.items()
    },
    "status": "DOC_PART1_REPOSITORY_INVENTORY_PASS"
}

output_bytes[AUDIT_OUT] = json_bytes(audit1)

OUT.mkdir(parents=True, exist_ok=True)

for path in OUTPUTS:
    if path.exists():
        raise FileExistsError(
            f"Refusing overwrite: {path}"
        )

created = []

try:
    # Reconfirm commit has not changed during execution.
    assert git_text("rev-parse", "HEAD") == head

    assert sha256(AUDIT6) == audit6_hash

    for path in OUTPUTS:
        with path.open("xb") as f:
            f.write(output_bytes[path])
        created.append(path)

    for path, data in output_bytes.items():
        assert sha256(path) == hashlib.sha256(
            data
        ).hexdigest()

    assert len(
        pd.read_csv(
            INVENTORY_OUT, encoding="utf-8-sig"
        )
    ) == len(inventory)

    assert read_json(AUDIT_OUT)["status"] == (
        "DOC_PART1_REPOSITORY_INVENTORY_PASS"
    )

except Exception:
    for path in created:
        if path.exists():
            path.unlink()
    raise

# ------------------------------------------------------------
# G. FINAL RESULTS
# ------------------------------------------------------------

print("\n" + "=" * 76)
print("07_DOCUMENTATION PART1 — FINAL AUDIT")
print("=" * 76)

print("Git commit            :", head[:12])
print("Tracked files         :", len(inventory))
print("Critical files        :", len(required_paths))
print("Dataset rows          : 348")
print("Methodological issues : 4 OPEN")
print("Academic status       : PROVISIONAL")
print("Source modifications  : NO")
print("Model training        : NO")

print("\nPHASE INVENTORY")
display(
    inventory.groupby("phase")
    .agg(
        files=("repository_path", "count"),
        total_size_bytes=("size_bytes", "sum")
    )
    .reset_index()
)

print("\nOUTPUT FILES")
for path in OUTPUTS:
    print("-", path)

print("\nFINAL STATUS:", audit1["status"])
print("=" * 76)


07_DOCUMENTATION PART1 — REPOSITORY PREFLIGHT
GIT REPOSITORY ROOT: PASS
Repository: C:\Penelitian Sentimen Bencana Nasional
GIT HEAD LOCK: PASS
Commit: ae5107c317eaee11a909251470be4b7503a6f9db
Branch: main
RESULT PART6 & LOCKED RESEARCH SOURCES: PASS
TRACKED GIT TREE INVENTORY: PASS
Total tracked files: 455

07_DOCUMENTATION PART1 — FINAL AUDIT
Git commit            : ae5107c317ea
Tracked files         : 455
Critical files        : 8
Dataset rows          : 348
Methodological issues : 4 OPEN
Academic status       : PROVISIONAL
Source modifications  : NO
Model training        : NO

PHASE INVENTORY


,phase,files,total_size_bytes
0,01_Collection,251,22907894
1,02_Preprocessing,11,1634875
2,03_Labeling,33,2571988
3,04_Modeling,89,4138364
4,05_Temporal,26,2365800
5,06_Result,29,1429234
6,07_Documentation,1,2
7,ROOT_OR_OTHER,15,377896



OUTPUT FILES
- C:\Penelitian Sentimen Bencana Nasional\07_Documentation\outputs\repository_inventory\repository_inventory.csv
- C:\Penelitian Sentimen Bencana Nasional\07_Documentation\outputs\repository_inventory\repository_inventory_summary.json
- C:\Penelitian Sentimen Bencana Nasional\07_Documentation\outputs\repository_inventory\documentation_blueprint.md
- C:\Penelitian Sentimen Bencana Nasional\07_Documentation\outputs\repository_inventory\part1_repository_inventory_audit.json

FINAL STATUS: DOC_PART1_REPOSITORY_INVENTORY_PASS


In [4]:
# 07_DOCUMENTATION — NOTEBOOK CLEANUP PREFLIGHT / CELL 02
# READ ONLY RESEARCH NOTEBOOKS
# STATIC AST + DUPLICATES + RISKS + SOURCE SHA256
# NO CELL EXECUTION | NO DELETION | NO SOURCE MODIFICATION
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
from collections import Counter, defaultdict
import ast
import hashlib
import io
import json
import re
import subprocess
import tokenize
import pandas as pd

ROOT = Path(r"C:\Penelitian Sentimen Bencana Nasional")
OUT = (
    ROOT / "07_Documentation" / "outputs"
    / "notebook_cleanup_preflight"
)

CELL_OUT = OUT / "notebook_cell_inventory.csv"
DEPENDENCY_OUT = OUT / "notebook_cell_dependencies.csv"
REPORT_OUT = OUT / "notebook_cleanup_review.md"
AUDIT_OUT = OUT / "notebook_cleanup_preflight_audit.json"

OUTPUTS = [
    CELL_OUT, DEPENDENCY_OUT, REPORT_OUT, AUDIT_OUT
]

def sha256_bytes(data):
    return hashlib.sha256(data).hexdigest()

def sha256(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024), b""
        ):
            h.update(chunk)
    return h.hexdigest()

def git(*args):
    p = subprocess.run(
        ["git", "-C", str(ROOT), *args],
        capture_output=True,
        check=True
    )
    return p.stdout

def rel(path):
    return path.relative_to(ROOT).as_posix()

def json_bytes(data):
    return json.dumps(
        data, ensure_ascii=False,
        indent=2, allow_nan=False
    ).encode("utf-8")

def src_text(cell):
    src = cell.get("source", [])
    return "".join(src) if isinstance(src, list) else str(src)

def classify_phase(path):
    first = Path(path).parts[0]
    if first == "01_Raw Data":
        return "01_Collection"
    if first.startswith("02_"):
        return "02_Cleaning"
    if first.startswith("03_"):
        return "03_Labeling"
    if first.startswith("04_"):
        return "04_Modeling"
    if first.startswith("05_"):
        return "05_Temporal"
    if first.startswith("06_"):
        return "06_Result"
    if first.startswith("07_"):
        return "07_Documentation"
    if first == "backup":
        return "HISTORICAL_BACKUP"
    return "OTHER"

def python_for_ast(source):
    """Replace IPython magics without running anything."""
    lines = []
    for line in source.splitlines():
        stripped = line.lstrip()
        if stripped.startswith(("!", "%", "?")):
            lines.append("pass  # IPython magic")
        elif stripped.startswith("%%"):
            lines.append("pass  # Cell magic")
        else:
            lines.append(line)
    return "\n".join(lines)

def analyze_ast(source):
    """Conservative Python name analysis; not runtime proof."""
    tree = ast.parse(python_for_ast(source))

    defined = set()
    loaded = set()

    for node in ast.walk(tree):
        if isinstance(node, (
            ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef
        )):
            defined.add(node.name)

        elif isinstance(node, ast.Name):
            if isinstance(node.ctx, (
                ast.Store, ast.Param
            )):
                defined.add(node.id)
            elif isinstance(node.ctx, ast.Load):
                loaded.add(node.id)

        elif isinstance(node, ast.Import):
            for alias in node.names:
                defined.add(
                    alias.asname or alias.name.split(".")[0]
                )

        elif isinstance(node, ast.ImportFrom):
            for alias in node.names:
                if alias.name != "*":
                    defined.add(alias.asname or alias.name)

    return defined, loaded

def source_signature(source):
    # Ignore whitespace and comments for candidate detection.
    try:
        stream = tokenize.generate_tokens(
            io.StringIO(source).readline
        )
        tokens = [
            t.string
            for t in stream
            if t.type not in {
                tokenize.COMMENT,
                tokenize.NL,
                tokenize.NEWLINE,
                tokenize.INDENT,
                tokenize.DEDENT,
                tokenize.ENDMARKER,
                tokenize.ENCODING
            }
        ]
        normalized = " ".join(tokens)
    except (tokenize.TokenError, IndentationError):
        normalized = source.strip()
    return sha256_bytes(normalized.encode("utf-8"))

print("=" * 74)
print("NOTEBOOK CLEANUP PREFLIGHT — READ ONLY")
print("=" * 74)

assert ROOT.is_dir()
assert (ROOT / ".git").exists()

OUT.mkdir(parents=True, exist_ok=True)

for path in OUTPUTS:
    if path.exists():
        raise FileExistsError(
            f"Existing output; no overwrite: {path}"
        )

HEAD = git("rev-parse", "HEAD").decode().strip()

# Enumerate tracked notebook paths from Git index.
tracked_paths = [
    p.decode("utf-8")
    for p in git("ls-files", "-z").split(b"\0")
    if p and p.decode("utf-8").endswith(".ipynb")
]

# Include new Documentation notebook even if untracked.
doc_notebook = "07_Documentation/07_documentation.ipynb"
if (ROOT / doc_notebook).is_file():
    tracked_paths.append(doc_notebook)

notebooks = sorted(set(tracked_paths))

assert notebooks, "No notebooks found"

print("Notebook candidates:", len(notebooks))

records = []
dependencies = []
source_hashes = {}
notebook_summaries = []
all_signatures = defaultdict(list)

for notebook_path in notebooks:
    path = (ROOT / notebook_path).resolve()

    assert path.is_relative_to(ROOT.resolve())
    assert path.is_file()
    assert not path.is_symlink()

    before = sha256(path)
    source_hashes[notebook_path] = before

    notebook = json.loads(path.read_text(encoding="utf-8"))

    assert notebook.get("nbformat") == 4, (
        f"Unexpected notebook format: {notebook_path}"
    )

    cells = notebook.get("cells")
    assert isinstance(cells, list)

    defs_per_cell = []
    loads_per_cell = []

    notebook_rows = []

    for index, cell in enumerate(cells, start=1):
        kind = cell.get("cell_type", "")
        source = src_text(cell)
        stripped = source.strip()

        is_empty = not stripped
        first_line = (
            next(
                (x.strip() for x in source.splitlines()
                 if x.strip()),
                ""
            )
        )

        parse_status = "NOT_CODE"
        defined, loaded = set(), set()

        if kind == "code":
            try:
                defined, loaded = analyze_ast(source)
                parse_status = "AST_PASS"
            except (SyntaxError, ValueError, TypeError):
                parse_status = "AST_REVIEW"

        defs_per_cell.append(defined)
        loads_per_cell.append(loaded)

        lower = source.lower()

        risk_api = bool(re.search(
            r"apify|\.actor\(|\.call\(|requests\.post"
            r"|execute_scraping|paid_run",
            lower
        ))
        risk_write = bool(re.search(
            r"\.to_csv\(|\.to_excel\(|\.to_json\("
            r"|write_text\(|write_bytes\(|open\(.{0,100}"
            r"""['"](?:w|a|x|wb|ab|xb)['"]""",
            source
        ))
        risk_delete = bool(re.search(
            r"\.unlink\(|shutil\.rmtree|os\.remove"
            r"|os\.unlink|\.rmdir\(",
            source
        ))
        risk_training = bool(re.search(
            r"\.fit\(|fit_transform\(|model\.train\("
            r"|\.backward\(|optimizer\.step\(",
            source
        ))

        diagnostic = bool(re.search(
            r"diagnostic|diag\b|debug|preview|smoke"
            r"|test.regression|regression.test",
            lower
        ))

        historical = (
            notebook_path.startswith("backup/")
            or bool(re.search(
                r"historical|legacy|backup",
                lower[:350]
            ))
        )

        audit_significant = bool(re.search(
            r"audit|reconcil|adjudicat|handoff"
            r"|integrity|final|lock|sha256",
            lower[:500]
        ))

        if kind == "markdown":
            recommendation = "KEEP"
            reason = "Documentation or notebook section heading"
        elif is_empty:
            recommendation = "REVIEW"
            reason = "Empty cell; inspect before removal"
        elif parse_status == "AST_REVIEW":
            recommendation = "REVIEW"
            reason = "Static parser could not fully analyze source"
        elif risk_api or risk_delete or risk_training:
            recommendation = "KEEP"
            reason = (
                "Execution/security-sensitive cell; "
                "must not be removed automatically"
            )
        elif diagnostic and not audit_significant:
            recommendation = "OPTIONAL"
            reason = "Diagnostic or preview candidate"
        elif historical:
            recommendation = "REVIEW"
            reason = "Historical or backup content"
        else:
            recommendation = "KEEP"
            reason = "Potential pipeline or research evidence"

        signature = source_signature(source) if stripped else ""

        row = {
            "notebook": notebook_path,
            "phase": classify_phase(notebook_path),
            "cell_index": index,
            "cell_type": kind,
            "cell_id": cell.get("id", ""),
            "first_line": first_line[:180],
            "source_lines": len(source.splitlines()),
            "source_chars": len(source),
            "source_sha256": sha256_bytes(
                source.encode("utf-8")
            ),
            "normalized_signature": signature,
            "execution_count": cell.get("execution_count"),
            "stored_output_count": len(cell.get("outputs", [])),
            "ast_status": parse_status,
            "defined_names": ";".join(sorted(defined)),
            "loaded_names": ";".join(sorted(loaded)),
            "possible_api_or_paid_call": risk_api,
            "possible_file_write": risk_write,
            "possible_deletion": risk_delete,
            "possible_training": risk_training,
            "diagnostic_keyword": diagnostic,
            "audit_significant_keyword": audit_significant,
            "recommendation": recommendation,
            "reason": reason,
            "duplicate_candidate": False,
            "downstream_reference_count": 0
        }

        notebook_rows.append(row)

        if stripped:
            all_signatures[
                (kind, signature)
            ].append((notebook_path, index))

    # Conservative cross-cell name dependency check.
    for target_index, target_loads in enumerate(
        loads_per_cell, start=1
    ):
        for source_index in range(
            1, target_index
        ):
            common = (
                defs_per_cell[source_index - 1]
                & target_loads
            )

            if common:
                dependencies.append({
                    "notebook": notebook_path,
                    "source_cell": source_index,
                    "target_cell": target_index,
                    "shared_names": ";".join(sorted(common)),
                    "shared_name_count": len(common),
                    "analysis_type": "STATIC_POSSIBLE_DEPENDENCY"
                })

    records.extend(notebook_rows)

    code_count = sum(
        c.get("cell_type") == "code" for c in cells
    )

    notebook_summaries.append({
        "notebook": notebook_path,
        "cells": len(cells),
        "code_cells": code_count,
        "markdown_cells": len(cells) - code_count,
        "sha256": before
    })

    assert sha256(path) == before, (
        f"Notebook changed during scan: {notebook_path}"
    )

# ------------------------------------------------------------
# D. DUPLICATE CANDIDATES (NO DELETION)
# ------------------------------------------------------------

duplicate_locations = set()

for key, locations in all_signatures.items():
    if len(locations) > 1:
        duplicate_locations.update(locations)

downstream_counts = Counter(
    (d["notebook"], d["source_cell"])
    for d in dependencies
)

for row in records:
    location = (row["notebook"], row["cell_index"])

    row["duplicate_candidate"] = (
        location in duplicate_locations
    )

    row["downstream_reference_count"] = (
        downstream_counts[location]
    )

    if row["duplicate_candidate"]:
        row["recommendation"] = "REVIEW"
        row["reason"] = (
            "Matching normalized source exists elsewhere; "
            "not sufficient evidence for deletion"
        )

    if (
        row["recommendation"] == "OPTIONAL"
        and row["downstream_reference_count"] > 0
    ):
        row["recommendation"] = "REVIEW"
        row["reason"] = (
            "Diagnostic candidate with possible "
            "downstream name dependencies"
        )

inventory = pd.DataFrame(records)
dep_df = pd.DataFrame(
    dependencies,
    columns=[
        "notebook", "source_cell", "target_cell",
        "shared_names", "shared_name_count",
        "analysis_type"
    ]
)

assert len(inventory) == sum(
    x["cells"] for x in notebook_summaries
)

assert not inventory["recommendation"].eq("REMOVE").any()

# ------------------------------------------------------------
# E. REPORT
# ------------------------------------------------------------

report_lines = [
    "# NOTEBOOK CLEANUP PREFLIGHT — REVIEW REPORT",
    "",
    f"**Git HEAD:** `{HEAD}`",
    f"**Notebooks scanned:** {len(notebooks)}",
    f"**Total cells:** {len(inventory)}",
    "",
    "## 1. Safety",
    "",
    "- No notebook cells were executed.",
    "- No notebook was rewritten.",
    "- No deletion recommendation is automatic.",
    "- Source hashes were rechecked after scanning.",
    "- Dependencies are static approximations, not proof.",
    "",
    "## 2. Notebook Summary",
    "",
    "| Notebook | Cells | Code | Markdown |",
    "|---|---:|---:|---:|"
]

for item in notebook_summaries:
    report_lines.append(
        f"| `{item['notebook']}` | {item['cells']} | "
        f"{item['code_cells']} | {item['markdown_cells']} |"
    )

report_lines.extend([
    "",
    "## 3. Classification Summary",
    "",
    "| Recommendation | Number of cells |",
    "|---|---:|"
])

for label, count in sorted(
    Counter(inventory["recommendation"]).items()
):
    report_lines.append(f"| {label} | {count} |")

report_lines.extend([
    "",
    "## 4. Review Candidates",
    "",
    "| Notebook | Cell | Category | Reason |",
    "|---|---:|---|---|"
])

candidates = inventory[
    inventory["recommendation"].isin(
        ["REVIEW", "OPTIONAL"]
    )
]

for row in candidates.itertuples(index=False):
    reason = str(row.reason).replace("|", "/")
    report_lines.append(
        f"| `{row.notebook}` | {row.cell_index} | "
        f"{row.recommendation} | {reason} |"
    )

report_lines.extend([
    "",
    "## 5. Interpretation",
    "",
    "An identical cell is not automatically redundant.",
    "Import, configuration, and validation cells may "
    "legitimately repeat across independent notebooks.",
    "",
    "A possible dependency means that a name defined "
    "in an earlier cell also appears in a later cell.",
    "The analysis is intentionally conservative and "
    "does not resolve all Python scopes or runtime state.",
    "",
    "## 6. Cleanup Approval Gate",
    "",
    "Do not remove, reorder, or merge a cell until:",
    "",
    "1. Its source and purpose have been reviewed.",
    "2. Downstream references have been checked.",
    "3. Generated research artifacts are identified.",
    "4. API/paid-operation and no-overwrite protections "
    "remain intact.",
    "5. A separate non-destructive cleanup plan is approved.",
    "",
    "**Next:** Manual cell-by-cell cleanup decision, "
    "followed by Documentation Part2.",
    ""
])

report = "\n".join(report_lines)

# ------------------------------------------------------------
# F. EXCLUSIVE OUTPUT AND FINAL AUDIT
# ------------------------------------------------------------

now = datetime.now(timezone.utc).isoformat()

output_bytes = {
    CELL_OUT: inventory.to_csv(
        index=False
    ).encode("utf-8-sig"),
    DEPENDENCY_OUT: dep_df.to_csv(
        index=False
    ).encode("utf-8-sig"),
    REPORT_OUT: report.encode("utf-8")
}

audit = {
    "stage": "07_Documentation",
    "part": "Notebook Cleanup Preflight",
    "generated_at_utc": now,
    "git_head": HEAD,
    "notebooks_scanned": len(notebooks),
    "cells_scanned": len(inventory),
    "possible_dependencies": len(dep_df),
    "duplicate_candidate_cells": int(
        inventory["duplicate_candidate"].sum()
    ),
    "classification_counts": dict(
        Counter(inventory["recommendation"])
    ),
    "notebook_sha256": source_hashes,
    "research_cells_executed": False,
    "research_notebooks_modified": False,
    "cells_deleted": 0,
    "automatic_removal_permitted": False,
    "manual_review_required": True,
    "analysis_limitations": [
        "AST does not prove runtime dependencies",
        "IPython magics require manual inspection",
        "Dynamic imports and notebook state may be missed",
        "Identical code may have distinct research roles",
        "Classification is conservative and heuristic"
    ],
    "output_sha256": {
        p.name: sha256_bytes(data)
        for p, data in output_bytes.items()
    },
    "status": "DOC_NOTEBOOK_CLEANUP_PREFLIGHT_PASS"
}

output_bytes[AUDIT_OUT] = json_bytes(audit)

created = []

try:
    for path in OUTPUTS:
        with path.open("xb") as f:
            f.write(output_bytes[path])
        created.append(path)

    for path, data in output_bytes.items():
        assert sha256(path) == sha256_bytes(data)

    for name, expected_hash in source_hashes.items():
        assert sha256(ROOT / name) == expected_hash, (
            f"Source changed after scan: {name}"
        )

except Exception:
    for path in created:
        if path.exists():
            path.unlink()
    raise

print("\n" + "=" * 74)
print("NOTEBOOK CLEANUP PREFLIGHT — FINAL AUDIT")
print("=" * 74)

print("Notebooks analyzed :", len(notebooks))
print("Cells analyzed     :", len(inventory))
print("Dependencies found :", len(dep_df))
print("Duplicate candidates:",
      int(inventory["duplicate_candidate"].sum()))
print("Cells deleted      : 0")

print("\nCLASSIFICATION")
display(
    inventory.groupby("recommendation")
    .size()
    .reset_index(name="cell_count")
)

print("\nNOTEBOOK SUMMARY")
display(pd.DataFrame(notebook_summaries))

print("\nOUTPUT FILES")
for path in OUTPUTS:
    print("-", path)

print("\nFINAL STATUS:", audit["status"])
print("=" * 74)


NOTEBOOK CLEANUP PREFLIGHT — READ ONLY
Notebook candidates: 10

NOTEBOOK CLEANUP PREFLIGHT — FINAL AUDIT
Notebooks analyzed : 10
Cells analyzed     : 238
Dependencies found : 1420
Duplicate candidates: 36
Cells deleted      : 0

CLASSIFICATION


,recommendation,cell_count
0,KEEP,157
1,OPTIONAL,1
2,REVIEW,80



NOTEBOOK SUMMARY


,notebook,cells,code_cells,markdown_cells,sha256
0,01_Raw Data/full_scraping_raw_data.ipynb,48,41,7,2e9a03232e9d1ca768aeb6ded64997387a6fe77e4b699e...
1,01_Raw Data/raw_collection.ipynb,21,14,7,11f66e07e4defe2b8d1c73212e2da8c886c5b4e2087f4b...
2,02_Cleaning/02_text_cleaning.ipynb,11,11,0,a24f8c0af62d3bddfed63652f0f7f1168cb8f62928579c...
3,03_Labeling/03_sentiment_labeling.ipynb,15,15,0,96c8ca7447437f6ab5f36288e480594806b0ad686cdad1...
4,04_Modeling/04_modeling.ipynb,38,38,0,35a64ecc5d3bc16a45a436363f365b1d5342b5fef3a601...
5,05_Temporal/05_temporal.ipynb,6,6,0,88fed1b5e4f3bb3cdf1d8a7bb971c7e6884e1fd0e13a80...
6,06_Result/06_result.ipynb,6,6,0,f0f0279051280c84fc4c6e2da432f481fc237f8e930805...
7,07_Documentation/07_documentation.ipynb,2,2,0,dce84408f48c5f915845e083b0c6a56aae5296e84ab569...
8,backup/full_scraping_raw_data_backup.ipynb,21,14,7,a9742922009083b16d1269cf56bf4070d27bdd6ade9435...
9,backup/raw_collection.ipynb,70,46,24,0e22c74429c543c35d6d22427dd7a80e23aba0c2dae5f6...



OUTPUT FILES
- C:\Penelitian Sentimen Bencana Nasional\07_Documentation\outputs\notebook_cleanup_preflight\notebook_cell_inventory.csv
- C:\Penelitian Sentimen Bencana Nasional\07_Documentation\outputs\notebook_cleanup_preflight\notebook_cell_dependencies.csv
- C:\Penelitian Sentimen Bencana Nasional\07_Documentation\outputs\notebook_cleanup_preflight\notebook_cleanup_review.md
- C:\Penelitian Sentimen Bencana Nasional\07_Documentation\outputs\notebook_cleanup_preflight\notebook_cleanup_preflight_audit.json

FINAL STATUS: DOC_NOTEBOOK_CLEANUP_PREFLIGHT_PASS
